In [1]:
:set -XOverloadedStrings
:set -XScopedTypeVariables
:set -XFlexibleContexts
:load ../lib/src/Quantale.hs ../lib/src/KanExtension.hs ../lib/src/Bitopos.hs ../lib/src/Distribution.hs ../lib/src/SubjectiveModel.hs
import Quantale
import KanExtension
import Bitopos
import Distribution
import SubjectiveModel
import Data.List (sortBy, maximumBy, minimumBy, intercalate)
import Data.Ord (comparing, Down(..))
import Data.Maybe (fromMaybe)
import Text.Printf (printf)
import System.IO (hSetBuffering, stdout, BufferMode(..))
hSetBuffering stdout LineBuffering
putStrLn "\x2705 SETUP OK: applied subjective modelling (Godel / Goguen / Luka scales from lib)"

✅ SETUP OK: applied subjective modelling (Godel / Goguen / Luka scales from lib)

# 🎚️ Субъективное Моделирование: Приложения

> **Апогей триптиха.** [SubjectiveModeling.ipynb](SubjectiveModeling.ipynb) ввёл меры правдоподобия $\mathrm{Pl}$ и доверия $\mathrm{Bel}$; [PytevIso.ipynb](PytevIso.ipynb) доказал, что все категорные представления этих мер изоморфны, и в §9 показал, что **вся теория параметризована кванталью** $L$. Здесь — расплата за страдания: вся тяжёлая математика превращается в простой инструмент. Мы берём одно достаточно богатое дело и проводим его через **три мира** — три варианта теории Пытьева (§1.9 его пособия-2022) — и смотрим, где они соглашаются, а где спорят.

**Девиз — Эрлангенская программа для неопределённости.** Феликс Клейн определил геометрию как теорию инвариантов группы преобразований: евклидова геометрия — инварианты движений, аффинная — инварианты аффинной группы, топология — инварианты гомеоморфизмов. **Точно так же вариант теории свидетельств = геометрия группы автоморфизмов шкалы $\mathrm{Aut}(L)$.** Чем меньше группа, тем больше содержательных инвариантов, тем «жёстче» мир. Наш сравнительный анализ — это буквально «что инвариантно во всех трёх геометриях» (мета-конструкция) против «что зависит от выбора мира» (артефакт шкалы).

**Практический итог:** одно и то же дело, один и тот же код (библиотека полиморфна по квантали) — три вердикта. Их совпадения — то, чему можно доверять безусловно; их расхождения показывают, какое свойство дела задело именно *вариантную* часть теории. Так мы отделяем мух от котлет — и тех, кто мне верит, от тех, кто нет.

## 📌 Содержание

| № | Раздел | Мир / статус |
|---|--------|--------------|
| 1 | Пролог: карта миров и Эрлангенская программа | обзор |
| 2 | Дело о пяти подозреваемых (и контрольный пример — двигатель) | постановка |
| 3 | Мир Гёделя $(\max, \min)$: слабейшее звено | ✅ вариант 1 |
| 4 | Мир Гогена $(\max, \cdot)$: накопление улик и лог-инварианты | ✅ вариант 3 |
| 5 | Зарубки: коллегия договаривается о порогах | ✅ вариант 2 |
| 6 | Мир Лукасевича (MV): вишенка — мост к вероятности | ✅ вариант 4 |
| 7 | Сравнительный анализ: мухи и котлеты | ✅ свод |

> **Как читать.** Разделы 3–6 — один и тот же код на разных шкалах (библиотека полиморфна). Смотрите не на абсолютные числа (они содержательны лишь до автоморфизма шкалы!), а на **порядок подозреваемых** и на то, **где он меняется между мирами**.

# 1. Пролог: Карта Миров

Один математический аппарат — sup-меры $\mathrm{Pl}(E) = \sup_{x\in E}\tau(x)$ и дуальные $\mathrm{Bel}$ — но **алгебра шкалы** $L$ бывает разной. Пытьев в §1.9 пособия-2022 описывает три варианта; в [PytevIso.ipynb](PytevIso.ipynb), §9 доказано, что это три подстановки квантали в одну мета-конструкцию. Библиотека курса реализует их как инстансы класса `Quantale` — и весь код ниже полиморфен.

| Мир | Кванталь $L$ | $\otimes$ (комбинирование улик) | $a \multimap b$ (кондиционирование) | $\mathrm{Aut}(L)$ | Характер |
|---|---|---|---|---|---|
| **Гёдель** (вар. 1) | $([0,1], \max, \min)$ | $\min$ — слабейшее звено | усечение | все строго монотонные биекции | осторожный, шумоустойчивый, «забывает» серию слабых улик |
| **Гоген** (вар. 3) | $([0,1], \max, \cdot)$ | $\cdot$ — накопление | деление | степенные $a \mapsto a^\alpha$ | психофизический; серия слабых улик топит; есть лог-инварианты |
| **Зарубки** (вар. 2) | Гёдель $+$ константы $S$ | $\min$ | усечение | фиксирующие $S$ | коллективный: договорённые пороги содержательны для всех |
| **Лукасевич** (вар. 4) | $([0,1], \max, \max(0,a{+}b{-}1))$ | граница Фреше | $\min(1,1{-}a{+}b)$ | только $\mathrm{id}$ | абсолютная шкала; ближе всех к вероятности (мост в §6) |

Диаграмма ниже раскладывает три *тензора комбинирования улик* $\otimes$ в один спектр. И это неожиданно красиво: тензор Гёделя ($\min$) — **верхняя граница Фреше** для вероятности конъюнкции $P(A\cap B)$, тензор Лукасевича ($\max(0,a{+}b{-}1)$) — **нижняя**, а произведение Гогена сидит ровно на независимости между ними. Три мира — не конкуренты, а координаты одной картины (подробно — §6). Сложение мер всюду одно и то же ($\max$), поэтому $\mathrm{Pl}$, носитель и нулевая зона **не зависят от выбора мира** — меняется только $\otimes$, то есть модель того, как складываются свидетельства. Вариант 2 (зарубки) отдельным тензором не является: это Гёдель с добавленными неподвижными точками, поэтому на спектре он делит место с Гёделем.

![Карта миров: спектр Фреше трёх тензоров](../diagrams/subj/subjapplied_worlds.svg)

**Инварианты и группа $\mathrm{Aut}(L)$ (Эрланген).** Клейн определил геометрию как теорию инвариантов группы преобразований. Здесь группа — это автоморфизмы шкалы $\mathrm{Aut}(L)$ (пересчёты, не меняющие теорию), а «геометрия» — то, что при них сохраняется. Чем меньше группа, тем больше содержательного:

- **Гёдель** — группа самая большая (все строго монотонные пересчёты), поэтому осмыслен **только порядок** подозреваемых;
- **зарубки** ($\Gamma_S$, фиксирует пороги $S$) — добавляется **положение относительно порогов** («выше разумного сомнения»);
- **Гоген** ($\{a\mapsto a^\alpha\}$, степенные) — добавляются **лог-отношения** $\log\mathrm{Pl}(A)/\log\mathrm{Pl}(B)$ («во сколько раз подозрительнее»);
- **Лукасевич** — группа тривиальна ($\{\mathrm{id}\}$), **все значения** абсолютны: это вероятностный предел.

Важная тонкость (её легко нарисовать неверно): $\Gamma_S$ и $\mathrm{Aut}(\text{Гоген})$ — **не** вложены друг в друга, а **несравнимы**. Степенная $a\mapsto a^\alpha$ фиксирует лишь $0$ и $1$, а зарубку $0.33$ — нет; наоборот, монотонный пересчёт, фиксирующий зарубки, не обязан быть степенным. Пересекаются они только по $\{\mathrm{id}\}$. Поэтому правильная картина — **не лестница, а ромб**: сверху общий предок $\mathrm{Aut}(\text{Гёдель})$, две несравнимые ветви уточнения, внизу — $\{\mathrm{id}\}$, где сходятся оба и рождается вероятность.

![Ромб групп автоморфизмов шкалы Aut(L)](../diagrams/subj/subjapplied_autladder.svg)

$0$ (невозможно) и $1$ (достоверно) неподвижны при *любой* $\gamma\in\mathrm{Aut}(L)$ — единственные абсолюты, общие всем мирам (см. врезку о нулевых множествах в [PytevIso.ipynb](PytevIso.ipynb), §8). Отсюда и стратегия анализа: **вывод, устойчивый во всех мирах, опирается только на порядок и нули — ему можно верить; вывод, что рушится при смене мира, держался на артефакте шкалы.**

# 2. Дело о Пяти Подозреваемых

Детектив-модельер расследует кражу. Круг сужен до пяти человек:

| Код | Подозреваемый | Базовое правдоподобие $\tau$ | Комментарий детектива |
|---|---|---|---|
| A | Alice | $0.9$ | мотив и возможность, видели рядом |
| B | Bob | $0.6$ | есть мотив, но алиби частичное |
| C | Carol | $0.6$ | подозрительное поведение |
| D | Dave | $0.3$ | слабый мотив |
| E | Erin | $0.0$ | **железное алиби — была в другом городе** |

Число $\tau(x)$ — субъективная оценка детектива, «насколько правдоподобна виновность $x$» в *ранговой* шкале: содержателен порядок, а не абсолютная величина (Гёдель). Erin с $\tau = 0$ образует **нулевую зону** $Z$: событие «виновна Erin» невозможно, и носитель дела — $X \setminus Z = \{A,B,C,D\}$ (см. врезку о нулевых множествах, [PytevIso.ipynb](PytevIso.ipynb), §8).

**Дуальная согласованность.** Доверие берём согласованным: $\bar\tau = \theta\circ\tau$ (в Гёделе $\theta(t) = 1-t$; в Лукасевиче эта инволюция окажется *внутренней* — §6). Тогда $\mathrm{Bel}(E) = \inf_{x\notin E}\bar\tau(x)$ — доверие к тому, что виновный внутри $E$.

**Три улики** приходят по ходу следствия; каждая задаёт *совместимость* $c_i(x) \in [0,1]$ подозреваемого $x$ с уликой:

| Улика | A | B | C | D | E |
|---|---|---|---|---|---|
| $c_1$: след ботинка 44-го размера | $0.9$ | $0.7$ | $0.5$ | $0.7$ | $0.4$ |
| $c_2$: синее волокно на месте | $0.5$ | $0.7$ | $0.8$ | $0.6$ | $0.3$ |
| $c_3$: замечен у дома в 21:00 | $0.6$ | $0.7$ | $0.4$ | $0.9$ | $0.2$ |

Комбинирование улики со свидетельством — тензор шкалы: $\tau'(x) = \tau(x) \otimes c_1(x) \otimes c_2(x) \otimes c_3(x)$. **Именно здесь миры расходятся сильнее всего:** в Гёделе $\otimes = \min$ (итог — слабейшее звено, серия улик не накапливается), в Гогене $\otimes = \cdot$ (улики перемножаются, серия слабых топит). Обратите внимание на Bob: три *средних* улики ($0.7$ каждая) — в Гёделе он останется на $0.6$, а в Гогене серия его утопит. Кто «подозрительнее» — зависит от мира; какой именно вопрос это выявит — увидим в §7.

**Контрольный пример (строгий) — диагностика двигателя.** Параллельно ведём технический пример из [SubjectiveModeling.ipynb](SubjectiveModeling.ipynb): НОЭ $\tilde x$ — неисправный узел, $\tau$ — правдоподобие по показаниям датчиков. Он даёт «сухую» проверку тех же формул там, где детективная фабула могла бы усыпить бдительность. Его числа появятся в сравнительной таблице §7.

# 3. Мир Гёделя: Слабейшее Звено

Первый вариант Пытьева — шкала $L = ([0,1], \max, \min)$. Это исходная теория возможностей: макситивные $\mathrm{Pl}$, минитивные $\mathrm{Bel}$, комбинирование улик через $\min$. Работаем полиморфной моделью `SubjModelQ UnitInterval` из библиотеки (`UnitInterval` — это и есть шкала Гёделя).

**Что вычисляем — весь инструментарий предыдущих ноутбуков в деле:**

- $\mathrm{Pl}(E) = \sup_{x\in E}\tau(x)$ и $\mathrm{Bel}(E) = \inf_{x\notin E}\bar\tau(x)$ — `smqPl`/`smqBel` (расширения Кана из ядра, [PytevIso.ipynb](PytevIso.ipynb), §1);
- **носитель** $X\setminus Z$ — главный фильтр достоверности (§8 PytevIso): Erin отсеивается;
- **решение** — подозреваемый максимального правдоподобия и «зазор разумного сомнения» $\mathrm{Pl}(\text{топ}) $ против $\mathrm{Pl}(\text{остальные})$;
- **комбинирование улик** через тензор $\otimes = \min$: `qTensor` на `UnitInterval`.

**Характер мира.** $\min$ — «слабейшее звено»: итоговое правдоподобие подозреваемого не ниже, чем его худшая совместимость с уликами, но и серию средних улик $\min$ **не накапливает** — три улики по $0.7$ дают ровно $0.7$. Отсюда осторожность и шумоустойчивость: одна ошибочная сильная улика не раздувает подозрение, но и «гора мелких косвенных» не давит. Ключевая деталь ниже — **тот же код** (`combine`) в §4 получит `goguen` вместо `ui`, и мир сменится без единой правки логики.

In [2]:
-- | Раздел 3: дело в мире Гёделя. Данные дела и полиморфная машинерия
--   определяются здесь ОДИН раз — разделы 4/6 переиспружают их, меняя лишь
--   инъекцию шкалы (ui -> goguen -> luka): один код, три мира.

suspects :: [Char]
suspects = "ABCDE"

nameOf :: Char -> String
nameOf c = fromMaybe [c] (lookup c
  [('A', "Alice"), ('B', "Bob"), ('C', "Carol"), ('D', "Dave"), ('E', "Erin")])

val :: [(Char, Double)] -> Char -> Double
val tab c = fromMaybe 0 (lookup c tab)

-- Базовое правдоподобие и три улики (совместимости) — общие для всех миров.
baseTab :: [(Char, Double)]
baseTab = [('A', 0.9), ('B', 0.6), ('C', 0.6), ('D', 0.3), ('E', 0.0)]

clues :: [(String, [(Char, Double)])]
clues =
  [ ("sled botinka 44", [('A', 0.9), ('B', 0.7), ('C', 0.5), ('D', 0.7), ('E', 0.4)])
  , ("sinee volokno",   [('A', 0.5), ('B', 0.7), ('C', 0.8), ('D', 0.6), ('E', 0.3)])
  , ("u doma v 21:00",  [('A', 0.6), ('B', 0.7), ('C', 0.4), ('D', 0.9), ('E', 0.2)]) ]

-- Полиморфная свёртка улик в квантали: tau(x) (x) c1(x) (x) c2(x) (x) c3(x).
combineWith :: Quantale q => (Double -> q) -> Char -> q
combineWith inj c = foldr qTensor (inj (val baseTab c)) [ inj (val tab c) | (_, tab) <- clues ]

-- Модель дела в инволютивной квантали (нужна для Bel-стороны).
caseModel :: InvolutiveQuantale q => (Double -> q) -> SubjModelQ q Char
caseModel inj = dualConsistentQ suspects (inj . val baseTab)

-- Ранжирование по убыванию: ПОРЯДОК — то, что инвариантно между мирами.
ranking :: Ord q => (Char -> q) -> [(Char, q)]
ranking f = sortBy (comparing (Down . snd)) [ (c, f c) | c <- suspects ]

-- --- Гёдель: шкала UnitInterval (max, min) ---
modelG :: SubjModelQ UnitInterval Char
modelG = caseModel ui

plG, belG :: [Char] -> Double
plG  e = unUI (smqPl modelG e)
belG e = unUI (smqBel modelG e)

demoGodel :: IO ()
demoGodel = do
  putStrLn "=== Мир Гёделя (max, min): тензор улик = min ==="
  putStrLn "-- Базовое правдоподобие (содержателен порядок, а не числа) --"
  mapM_ (\(c, v) -> printf "  %-6s τ = %.2f\n" (nameOf c) (unUI v))
        (ranking (ui . val baseTab))
  printf "  Носитель (X \\ Z, нулевая зона Erin отсечена): %s\n"
         (intercalate ", " [ nameOf c | c <- suspects, val baseTab c > 0 ])
  putStrLn "-- Меры событий  Pl / Bel --"
  let evs = [ ("{Alice}", "A"), ("{Alice,Bob}", "AB")
            , ("виновный не Erin (ABCD)", "ABCD"), ("{Erin}", "E") ]
  mapM_ (\(nm, e) -> printf "  %-26s Pl = %.2f   Bel = %.2f\n" nm (plG e) (belG e)) evs
  putStrLn "-- После трёх улик:  τ ⊗ c1 ⊗ c2 ⊗ c3  (в Гёделе = min) --"
  mapM_ (\(c, v) -> printf "  %-6s → %.2f\n" (nameOf c) (unUI v)) (ranking (combineWith ui))
  putStrLn "  (!) Bob: три средние улики 0.7 НЕ накапливаются (min = 0.6);"
  putStrLn "      Alice тонет до 0.5 из-за слабого звена (синее волокно 0.5)."
  let scored = [ (c, unUI (combineWith ui c)) | c <- suspects ]
      (best, bv) = maximumBy (comparing snd) scored
      rest = maximum [ v | (c, v) <- scored, c /= best ]
  printf "  Решение Гёделя: %s (Pl=%.2f), зазор разумного сомнения: %.2f\n"
         (nameOf best) bv (bv - rest)

demoGodel

=== Мир Гёделя (max, min): тензор улик = min ===
-- Базовое правдоподобие (содержателен порядок, а не числа) --
  Alice  τ = 0.90
  Bob    τ = 0.60
  Carol  τ = 0.60
  Dave   τ = 0.30
  Erin   τ = 0.00
  Носитель (X \ Z, нулевая зона Erin отсечена): Alice, Bob, Carol, Dave
-- Меры событий  Pl / Bel --
  {Alice}                    Pl = 0.90   Bel = 0.40
  {Alice,Bob}                Pl = 0.90   Bel = 0.40
  виновный не Erin (ABCD)    Pl = 0.90   Bel = 1.00
  {Erin}                     Pl = 0.00   Bel = 0.10
-- После трёх улик:  τ ⊗ c1 ⊗ c2 ⊗ c3  (в Гёделе = min) --
  Bob    → 0.60
  Alice  → 0.50
  Carol  → 0.40
  Dave   → 0.30
  Erin   → 0.00
  (!) Bob: три средние улики 0.7 НЕ накапливаются (min = 0.6);
      Alice тонет до 0.5 из-за слабого звена (синее волокно 0.5).
  Решение Гёделя: Bob (Pl=0.60), зазор разумного сомнения: 0.10

# 4. Мир Гогена: Накопление Улик и Лог-Инварианты

Третий вариант Пытьева — шкала $L = ([0,1], \max, \cdot)$: сложение мер всё то же $\max$ (поэтому $\mathrm{Pl}$, носитель, $\Lambda$ **не меняются** — см. «мухи и котлеты», [PytevIso.ipynb](PytevIso.ipynb), §9), а вот **тензор комбинирования улик — обычное умножение**. Меняется ровно одна строка кода: `combineWith goguen` вместо `combineWith ui`.

**Накопление.** $\cdot$ не идемпотентна ($a\cdot a < a$): серия улик перемножается. Три средние улики $0.7$ у Bob дают $0.7^3 \approx 0.34$ на его базу — серия *топит*. Alice с сильной базой ($0.9$) и сильным следом ботинка ($0.9$) держится, несмотря на одно слабое звено. **Порядок подозреваемых переворачивается: Гёдель арестовывал Bob, Гоген — Alice.** Это не спор о числах, а разная физика свидетельств: «слабейшее звено» против «накопления». Который прав — зависит от того, независимы ли улики (тогда прав Гоген) или это одна и та же информация под разными углами (тогда прав Гёдель). Вариант шкалы — это модельное решение о природе улик.

**Лог-инварианты (новинка Гогена).** $\mathrm{Aut}(\text{Гоген}) = \{a\mapsto a^\alpha\}$ — степенные (психофизические) пересчёты. Инвариант группы — **отношение логарифмов** $\pi = \log\mathrm{Pl}(A)/\log\mathrm{Pl}(B)$: под $a\mapsto a^\alpha$ оба логарифма умножаются на $\alpha$, отношение не меняется. Впервые содержательно «во сколько раз (в лог-шкале) один подозреваемее другого» — в Гёделе такой вопрос был бессмыслицей (там инвариантен только порядок).

**Bel-этаж — деквантование Маслова.** У Гогена нет самодуальной инволюции (в библиотеке нет `InvolutiveQuantale Goguen` — это теорема, выраженная типом: `dualConsistentQ goguen` не скомпилируется). Двойственность уезжает в **тропическую шкалу** $([0,\infty], \min, +)$ через $v = -\log u$. Величина $-\log\tau(x)$ — «**энергия невиновности**»: чем меньше, тем подозрительнее; alibi Erin ($\tau=0$) даёт бесконечную энергию. Комбинирование улик = *сложение* энергий (min-plus), а «наиболее правдоподобный» = «наименьшей энергии» — это буквально принцип наименьшего действия и мост к большим уклонениям (тропа 4).

In [3]:
-- | Раздел 4: то же дело в мире Гогена. Меняется ТОЛЬКО инъекция шкалы:
--   combineWith goguen вместо combineWith ui. Логика (combineWith/ranking) — та же.

-- Форматирование тропической «энергии» (inf для нулевой зоны).
fmtE :: Double -> String
fmtE v = if isInfinite v then "  inf" else printf "%.3f" v

demoGoguen :: IO ()
demoGoguen = do
  putStrLn "=== Мир Гогена (max, *): тензор улик = умножение ==="
  putStrLn "-- После трёх улик:  τ · c1 · c2 · c3  (накопление) --"
  mapM_ (\(c, v) -> printf "  %-6s → %.3f\n" (nameOf c) (unGoguen v)) (ranking (combineWith goguen))
  putStrLn "  (!) Bob: 0.6 · 0.7³ = 0.206 — серия утопила; Alice держится на сильной базе."
  let ordG = map (nameOf . fst) (ranking (combineWith ui))
      ordP = map (nameOf . fst) (ranking (combineWith goguen))
  printf "  Порядок Гёделя: %s\n" (intercalate " > " ordG)
  printf "  Порядок Гогена: %s\n" (intercalate " > " ordP)
  putStrLn "  (!!) Ранг перевернулся: Гёдель арестует Bob, Гоген — Alice."
  putStrLn "-- Лог-инвариант  π = log Pl(A) / log Pl(B)  (инвариант к a ↦ a^α) --"
  let pv c = unGoguen (combineWith goguen c)
      piAB a = logBase 10 (pv 'A' ** a) / logBase 10 (pv 'B' ** a)
  printf "  π(Alice/Bob): α=1 → %.4f,  α=2.5 → %.4f  (одинаков: лог-инвариант)\n"
         (piAB 1) (piAB 2.5)
  putStrLn "-- Деквантование Маслова: энергия невиновности  E(x) = -log τ(x) --"
  let energy c = unTrop (dequant (combineWith goguen c))
  mapM_ (\(c, _) -> printf "  %-6s E = %s\n" (nameOf c) (fmtE (energy c)))
        (ranking (combineWith goguen))
  putStrLn "  Наименьшая энергия = наиболее подозреваемый; Erin = inf (железное алиби)."
  let byEnergy = map nameOf (sortBy (comparing energy) suspects)
  printf "  Проверка изо (-log): порядок по энергии == порядок Гогена:  %s\n"
         (show (byEnergy == ordP))

demoGoguen

=== Мир Гогена (max, *): тензор улик = умножение ===
-- После трёх улик:  τ · c1 · c2 · c3  (накопление) --
  Alice  → 0.243
  Bob    → 0.206
  Dave   → 0.113
  Carol  → 0.096
  Erin   → 0.000
  (!) Bob: 0.6 · 0.7³ = 0.206 — серия утопила; Alice держится на сильной базе.
  Порядок Гёделя: Bob > Alice > Carol > Dave > Erin
  Порядок Гогена: Alice > Bob > Dave > Carol > Erin
  (!!) Ранг перевернулся: Гёдель арестует Bob, Гоген — Alice.
-- Лог-инвариант  π = log Pl(A) / log Pl(B)  (инвариант к a ↦ a^α) --
  π(Alice/Bob): α=1 → 0.8949,  α=2.5 → 0.8949  (одинаков: лог-инвариант)
-- Деквантование Маслова: энергия невиновности  E(x) = -log τ(x) --
  Alice  E = 1.415
  Bob    E = 1.581
  Dave   E = 2.177
  Carol  E = 2.343
  Erin   E =   inf
  Наименьшая энергия = наиболее подозреваемый; Erin = inf (железное алиби).
  Проверка изо (-log): порядок по энергии == порядок Гогена:  True

# 5. Зарубки: Коллегия Договаривается о Порогах

Второй вариант Пытьева (§1.9.1) — шкала Гёделя, но с выделенными **неподвижными точками** $S = \{a_1, \dots, a_n\}$, которые все члены коллегии условились понимать содержательно. Группа пересчётов сужается до $\Gamma_S$ (автоморфизмы, фиксирующие $S$), и появляется новый слой инвариантов: не только порядок, но и **положение относительно порогов**.

**Зачем это присяжным.** У каждого модельера-эксперта своя личная шкала ощущения правдоподобия (связаны они автоморфизмом $\gamma\in\Gamma$). В чистом Гёделе они согласны лишь о порядке. Но людям проще договориться, когда в шкале есть общие «зарубки» помимо $0$ и $1$: договоримся, что $a_1 = 0.33$ — «тень сомнения», $a_2 = 0.66$ — «серьёзное подозрение». Тогда **вопрос «выше ли подозреваемый порога серьёзности» имеет один ответ для всей коллегии** — он $\Gamma_S$-инвариантен, даже если личные числа присяжных различаются.

**Проекторы.** Пытьев формализует огрубление до зон операторами $(u)^\smile_a = a\vee u$ (сдвиг снизу) и $(u)^\frown_a = a\otimes u$ (поджатие сверху); в библиотеке — `projUp`, `projDown`, `clampScale` (см. [PytevIso.ipynb](PytevIso.ipynb), §9, где показано, что они коммутируют с pl-интегралом — универсальны для любой квантали). `clampScale lo hi` зажимает значение в неподвижный интервал $[lo, hi]$; для Гёделя это обычный $\min(hi, \max(lo, u))$. Проекция раскладывает шкалу в цепочку зон $[0, a_1], [a_1, a_2], [a_2, 1]$, и внутри зоны различия «стираются» — остаётся лишь принадлежность зоне, общая для всех.

Ниже — проверка $\Gamma_S$-инвариантности: два присяжных с *разными* личными шкалами (связанными $\gamma_S$, фиксирующим зарубки) получают **одни и те же зоны** для всех подозреваемых. Это операционализация коллективной экспертизы — согласие без навязывания единой шкалы.

In [4]:
-- | Раздел 5: вариант 2 — зарубки коллегии. Шкала Гёделя + константы S={0.33,0.66}.
--   Проекторы из библиотеки (projUp/projDown/clampScale); Gamma_S-инвариантность зон.

notchLo, notchHi :: Double
notchLo = 0.33
notchHi = 0.66

-- Зона подозрения по зарубкам (Gamma_S-инвариант).
band :: Double -> String
band v
  | v < notchLo = "тень сомнения"
  | v < notchHi = "подозрение"
  | otherwise   = "серьёзное подозрение"

-- Личная шкала другого присяжного: gamma_S фиксирует 0, 0.33, 0.66, 1
--   (автоморфизм из Gamma_S — гнёт значения ВНУТРИ зон, не пересекая зарубки).
gammaS :: Double -> Double
gammaS x
  | x <= notchLo = notchLo * (x / notchLo) ** 1.7
  | x <= notchHi = notchLo + (notchHi - notchLo) * ((x - notchLo) / (notchHi - notchLo)) ** 0.6
  | otherwise    = notchHi + (1 - notchHi) * ((x - notchHi) / (1 - notchHi)) ** 1.3

demoNotches :: IO ()
demoNotches = do
  putStrLn "=== Вариант 2: зарубки коллегии (Гёдель + S = {0.33, 0.66}) ==="
  putStrLn "-- Зона каждого подозреваемого (содержательна для всей коллегии) --"
  mapM_ (\c -> printf "  %-6s τ = %.2f  →  %s\n" (nameOf c) (val baseTab c) (band (val baseTab c)))
        suspects
  printf "  γ_S фиксирует зарубки: γ(0.33) = %.3f,  γ(0.66) = %.3f\n"
         (gammaS notchLo) (gammaS notchHi)
  let agree = all (\c -> band (val baseTab c) == band (gammaS (val baseTab c))) suspects
  printf "  Два присяжных (личные шкалы связаны γ_S) согласны о зонах:  %s\n" (show agree)
  putStrLn "-- Проектор clampScale в неподвижный интервал [0.33, 0.66] (библиотека) --"
  mapM_ (\c -> printf "  %-6s → %.2f\n" (nameOf c)
                 (unUI (clampScale (ui notchLo) (ui notchHi) (ui (val baseTab c)))))
        suspects
  putStrLn "  (Alice поджата до 0.66, Dave/Erin подняты до 0.33 — внутри зон различия стёрты.)"

demoNotches

=== Вариант 2: зарубки коллегии (Гёдель + S = {0.33, 0.66}) ===
-- Зона каждого подозреваемого (содержательна для всей коллегии) --
  Alice  τ = 0.90  →  серьёзное подозрение
  Bob    τ = 0.60  →  подозрение
  Carol  τ = 0.60  →  подозрение
  Dave   τ = 0.30  →  тень сомнения
  Erin   τ = 0.00  →  тень сомнения
  γ_S фиксирует зарубки: γ(0.33) = 0.330,  γ(0.66) = 0.660
  Два присяжных (личные шкалы связаны γ_S) согласны о зонах:  True
-- Проектор clampScale в неподвижный интервал [0.33, 0.66] (библиотека) --
  Alice  → 0.66
  Bob    → 0.60
  Carol  → 0.60
  Dave   → 0.33
  Erin   → 0.33
  (Alice поджата до 0.66, Dave/Erin подняты до 0.33 — внутри зон различия стёрты.)

# 6. Мир Лукасевича: Вишенка — Мост к Вероятности

Четвёртый вариант — тот, что мета-конструкция §9 [PytevIso.ipynb](PytevIso.ipynb) **предсказала до того, как его кто-либо строил**. Шкала $L = ([0,1], \max, \otimes_\text{Ł})$ с $a\otimes_\text{Ł}b = \max(0, a+b-1)$ — MV-алгебра, алгебра нечёткой логики Лукасевича. У неё два свойства, которых нет ни у одного из трёх вариантов Пытьева, и оба ведут к вероятности.

**1. Инволюция внутренняя.** У Гёделя двойственность $\theta(t)=1-t$ приходилось постулировать извне (класс $\Theta$): residuation-отрицание $a\multimap 0$ вырождается в $[a=0]$ и инволюцией не является. У Лукасевича $a\multimap 0 = 1-a$ — и это **настоящая инволюция** ($(a\multimap 0)\multimap 0 = a$). Значит $\mathrm{Bel}$ выводится из $\mathrm{Pl}$ **внутри теории**, без внешнего довеска: `dualConsistentQ luka` компилируется (в отличие от Гогена — там инстанса `InvolutiveQuantale` нет), потому что тип `Luka` — инстанс `InvolutiveQuantale` с `inv a = a ⊸ 0`. Двойственность стала теоремой, а не соглашением.

**2. Границы Фреше — и вилка вокруг вероятности.** $a\otimes_\text{Ł}b = \max(0, a+b-1)$ — это в точности **нижняя граница Фреше** для $P(A\cap B)$ при неизвестной зависимости, а $\min(a,b)$ (тензор Гёделя!) — **верхняя**. Отсюда неожиданное единство: **Гёдель и Лукасевич — это два конца вилки, в которой лежит любая вероятность конъюнкции.** Гёдель говорит «в лучшем случае», Лукасевич — «в худшем»; независимость (перемножение, Гоген) сидит между ними. Три мира оказались не конкурентами, а координатами одной картины.

**3. Вероятность = пространство состояний (Крупа–Панти).** Ключевая теорема теории MV-алгебр: **состояния** на MV-алгебре — нормированные аддитивные функционалы — это в точности интегралы по борелевским вероятностным мерам (Kroupa 2006, Panti 2008). А теорема **Мундичи** (1986) говорит, что MV-алгебры — это единичные интервалы решёточно упорядоченных абелевых групп: арифметика сложения встроена в саму шкалу. Вместе: **вероятность не альтернатива четвёртому миру, а его „измерительный слой“** — пространство состояний. Ниже мы строим вероятностную модель того же дела и показываем, что она возникает как состояние MV-алгебры: аддитивность (не $\max$!), нормировка, ожидание. Твоё давнее подозрение — «Лукасевич очень похож на вероятность» — оказывается теоремой; переход к $\sigma$-алгебрам, о котором ты говорил, — это $\sigma$-полные MV и теорема Лумиса–Сикорского. Вишенка на категорном торте.

In [5]:
-- | Раздел 6: мир Лукасевича. Внутренняя инволюция, границы Фреше, мост к вероятности.

-- Внутренняя инволюция Лукасевича: inv a = a ⊸ 0 = 1 - a.
invL :: Double -> Double
invL a = unLuka (inv (luka a))

-- «Инволюция» Гёделя через residuation вырождается: a ⊸ 0 = [a = 0].
invGodel :: Double -> Double
invGodel a = unUI (qHom (ui a) lbot)

-- Модель дела в MV-шкале: dualConsistentQ компилируется (Luka инволютивна).
modelL :: SubjModelQ Luka Char
modelL = caseModel luka

plL, belL :: [Char] -> Double
plL  e = unLuka (smqPl modelL e)
belL e = unLuka (smqBel modelL e)

-- Вероятностная модель того же дела: нормировка базового правдоподобия.
probOf :: Char -> Double
probOf c = val baseTab c / sum (map (val baseTab) suspects)

prEvent :: [Char] -> Double
prEvent e = sum [ probOf c | c <- e ]

-- Состояние на MV-алгебре = интеграл MV-функции по вероятности (Крупа–Панти).
stateMV :: (Char -> Double) -> Double
stateMV f = sum [ probOf c * f c | c <- suspects ]

demoLuka :: IO ()
demoLuka = do
  putStrLn "=== Мир Лукасевича (max, ⊗_Ł = max(0, a+b−1)): MV-алгебра ==="
  putStrLn "-- Внутренняя инволюция: Bel получается из Pl БЕЗ внешнего Θ --"
  printf "  Лукасевич:  a=0.375 → inv=%.3f → inv∘inv=%.3f  (тождество — инволюция внутренняя)\n"
         (invL 0.375) (invL (invL 0.375))
  printf "  Гёдель:     a=0.375 → inv=%.3f → inv∘inv=%.3f  (коллапс [a>0] — инволюции нет)\n"
         (invGodel 0.375) (invGodel (invGodel 0.375))
  printf "  Bel(виновный не Erin) = %.2f,  Pl({Erin}) = %.2f  (двойственность внутренняя)\n"
         (belL "ABCD") (plL "E")
  putStrLn "-- Границы Фреше: Гёдель и Лукасевич — два конца вилки для P(A∩B) --"
  let a = 0.6; b = 0.7
      lo = unLuka (qTensor (luka a) (luka b))
      hi = unUI  (qTensor (ui a)   (ui b))
      indep = a * b
  printf "  P(A)=%.1f, P(B)=%.1f:  вилка Фреше [%.2f (Лук., worst-case) .. %.2f (Гёдель, best-case)], независимость %.2f\n"
         a b lo hi indep
  putStrLn "-- Вероятность = пространство состояний 4-го мира (Крупа–Панти) --"
  mapM_ (\c -> printf "  %-6s Pr = %.3f\n" (nameOf c) (probOf c)) suspects
  printf "  Аддитивность (не max!):  Pr({Alice,Bob}) = %.3f = Pr(A)+Pr(B);  но Pl({Alice,Bob}) = %.2f\n"
         (prEvent "AB") (plL "AB")
  printf "  Состояние s(χ_{Alice,Bob}) = %.3f = Pr,  s(1) = %.3f (нормировка) — это и есть вероятность\n"
         (stateMV (\c -> if c `elem` "AB" then 1 else 0)) (stateMV (const 1))
  putStrLn "  ⇒ вероятность живёт ВНУТРИ теории Лукасевича как её состояния (Мундичи: MV = интервалы ℓ-групп)."

demoLuka

=== Мир Лукасевича (max, ⊗_Ł = max(0, a+b−1)): MV-алгебра ===
-- Внутренняя инволюция: Bel получается из Pl БЕЗ внешнего Θ --
  Лукасевич:  a=0.375 → inv=0.625 → inv∘inv=0.375  (тождество — инволюция внутренняя)
  Гёдель:     a=0.375 → inv=0.000 → inv∘inv=1.000  (коллапс [a>0] — инволюции нет)
  Bel(виновный не Erin) = 1.00,  Pl({Erin}) = 0.00  (двойственность внутренняя)
-- Границы Фреше: Гёдель и Лукасевич — два конца вилки для P(A∩B) --
  P(A)=0.6, P(B)=0.7:  вилка Фреше [0.30 (Лук., worst-case) .. 0.60 (Гёдель, best-case)], независимость 0.42
-- Вероятность = пространство состояний 4-го мира (Крупа–Панти) --
  Alice  Pr = 0.375
  Bob    Pr = 0.250
  Carol  Pr = 0.250
  Dave   Pr = 0.125
  Erin   Pr = 0.000
  Аддитивность (не max!):  Pr({Alice,Bob}) = 0.625 = Pr(A)+Pr(B);  но Pl({Alice,Bob}) = 0.90
  Состояние s(χ_{Alice,Bob}) = 0.625 = Pr,  s(1) = 1.000 (нормировка) — это и есть вероятность
  ⇒ вероятность живёт ВНУТРИ теории Лукасевича как её состояния (Мундичи: MV = интервалы ℓ-г

# 7. Сравнительный Анализ: Мухи и Котлеты

Одно дело, один код (библиотека полиморфна по квантали), четыре мира. Пора свести вердикты и отделить то, чему можно верить безусловно, от того, что держится на выборе шкалы.

**Таблица вердиктов** (ячейка ниже строит её из тех же `combineWith`):

- **Гёдель** ($\min$, слабейшее звено) арестует **Bob**: Alice утонула на слабом звене (синее волокно $0.5$).
- **Гоген** ($\cdot$, накопление) арестует **Alice**: её сильная база и след ботинка перемножились, серия средних улик Bob'а не догнала.
- **Лукасевич** (Фреше, worst-case) **не обвинит никого**: агрессивная конъюнкция $\max(0,\Sigma-1)$ схлопывает набор косвенных улик в $0$ — «оправдать за недостаточностью». Самый осторожный мир, и это его добродетель.

**Инвариантное — мета-уровень, общий всем геометриям (Эрланген):**

- **порядок базовой оценки** $\text{Alice} > \text{Bob} = \text{Carol} > \text{Dave} > \text{Erin}$ одинаков во всех мирах (сложение мер всюду $\max$ — $\mathrm{Pl}$ и $\Lambda$ не зависят от $\otimes$);
- **нулевая зона**: $\mathrm{Pl}(\{\text{Erin}\}) = 0$ и носитель $X\setminus\{\text{Erin}\}$ — во всех мирах (единственные абсолюты шкалы — $0$ и $1$);
- **$\mathrm{Bel}(\text{виновный не Erin}) = 1$** — во всех мирах.

Этим выводам можно верить безусловно: они опираются только на порядок и нули — на то, что инвариантно относительно $\mathrm{Aut}(L)$.

**Вариантное — артефакт выбора шкалы:**

- **ранг после комбинирования улик** переворачивается между мирами, потому что зависит от тензора $\otimes$: $\min$ (одна и та же информация под разными углами) против $\cdot$ (независимые улики) против Фреше (худший случай зависимости). Это не факт о деле, а **модельное решение о природе улик** — и его нужно принимать сознательно.

**Мораль — и ответ на исходный вопрос «отделить мух от котлет».** Вывод, устойчивый во всех мирах, стои́т на инвариантах — ему верь. Вывод, что рушится при смене мира, держался на выборе $\otimes$ — не прячь это в одно число, как сделала бы вероятность, а **вскрой развилку**: «если улики независимы — Alice; если это одна информация — Bob; если нужна железобетонная конъюнкция — никто». Честная неопределённость вместо ложной точности. Именно в этом обещание субъективного моделирования, и категорный взгляд (теория = геометрия группы $\mathrm{Aut}(L)$) делает его *инструментом*, а не 400-страничным лабиринтом: выбор мира — это выбор геометрии, а не шаманство.

In [6]:
-- | Раздел 7: сравнительный анализ. Один combineWith — три мира — таблица вердиктов.

demoCompare :: IO ()
demoCompare = do
  putStrLn "=== Сравнительный анализ: один и тот же код, три мира ==="
  let gs c = unUI     (combineWith ui c)
      ps c = unGoguen (combineWith goguen c)
      ls c = unLuka   (combineWith luka c)
  putStrLn "  Подозреваемый |  Гёдель | Гоген  | Лукасевич"
  putStrLn "  ──────────────┼─────────┼────────┼──────────"
  mapM_ (\c -> printf "  %-13s |  %.2f   | %.3f  |  %.2f\n" (nameOf c) (gs c) (ps c) (ls c)) suspects
  let dec f  = nameOf (fst (maximumBy (comparing snd) [ (c, f c) | c <- suspects ]))
      best f = maximum [ f c | c <- suspects ]
  printf "  Решение       |  %-6s | %-6s | %s\n"
         (dec gs) (dec ps) (if best ls <= 0 then "— (оправдать)" else dec ls)
  putStrLn ""
  putStrLn "-- Инвариантное (во всех мирах — можно верить безусловно) --"
  printf "  • Порядок базовой оценки: %s\n"
         (intercalate " > " (map (nameOf . fst) (ranking (ui . val baseTab))))
  putStrLn "  • Нулевая зона: Pl({Erin}) = 0 всюду; носитель = X ∖ {Erin}"
  putStrLn "  • Bel(виновный не Erin) = 1 всюду"
  putStrLn "-- Вариантное (артефакт выбора шкалы — на этом вердикт строить нельзя) --"
  putStrLn "  • Ранг ПОСЛЕ комбинирования: Гёдель → Bob, Гоген → Alice, Лукасевич → никто."
  putStrLn "    Причина — тензор ⊗: min (одна информация) / · (независимость) / Фреше (worst-case)."
  putStrLn "  ⇒ Устойчивое во всех мирах опирается на порядок и нули — верь ему."
  putStrLn "    Рушащееся при смене мира держалось на выборе ⊗ — вскрой развилку, не прячь в число."

demoCompare

=== Сравнительный анализ: один и тот же код, три мира ===
  Подозреваемый |  Гёдель | Гоген  | Лукасевич
  ──────────────┼─────────┼────────┼──────────
  Alice         |  0.50   | 0.243  |  0.00
  Bob           |  0.60   | 0.206  |  0.00
  Carol         |  0.40   | 0.096  |  0.00
  Dave          |  0.30   | 0.113  |  0.00
  Erin          |  0.00   | 0.000  |  0.00
  Решение       |  Bob    | Alice  | — (оправдать)

-- Инвариантное (во всех мирах — можно верить безусловно) --
  • Порядок базовой оценки: Alice > Bob > Carol > Dave > Erin
  • Нулевая зона: Pl({Erin}) = 0 всюду; носитель = X ∖ {Erin}
  • Bel(виновный не Erin) = 1 всюду
-- Вариантное (артефакт выбора шкалы — на этом вердикт строить нельзя) --
  • Ранг ПОСЛЕ комбинирования: Гёдель → Bob, Гоген → Alice, Лукасевич → никто.
    Причина — тензор ⊗: min (одна информация) / · (независимость) / Фреше (worst-case).
  ⇒ Устойчивое во всех мирах опирается на порядок и нули — верь ему.
    Рушащееся при смене мира держалось на выборе ⊗

---

**Прикладная сноска** к триптиху [SubjectiveModeling.ipynb](SubjectiveModeling.ipynb) → [PytevIso.ipynb](PytevIso.ipynb) → (этот ноутбук). Библиотека: [darklordshish/SubjectiveModeling](https://github.com/darklordshish/SubjectiveModeling). ↩ [Путеводитель](../README.ipynb)